### <font style="color:blue">Project 2: Kaggle Competition - Classification</font>

#### Maximum Points: 100

<div>
    <table><h5>
        <tr><td>Sr. no.</td> <td>Section</td> <td>Points</td> </tr>
        <tr><td>1</td> <td>Data Loader</td> <td>10</td> </tr>
        <tr><td>2</td> <td>Configuration</td> <td>5</td> </tr>
        <tr><td>3</td> <td>Evaluation Metric</td> <td>10</td> </tr>
        <tr><td>4</td> <td>Train and Validation</td> <td>5</td> </tr>
        <tr><td>5</td> <td>Model</td> <td>5</td> </tr>
        <tr><td>6</td> <td>Utils</td> <td>5</td> </tr>
        <tr><td>7</td> <td>Experiment</td><td>5</td> </tr>
        <tr><td>8</td> <td>TensorBoard Dev Scalars Log Link</td> <td>5</td> </tr>
        <tr><td>9</td> <td>Kaggle Profile Link</td> <td>50</td> </tr>
    </h5></table>
</div>


In [1]:
from IPython.display import display, HTML
from IPython.core.interactiveshell import InteractiveShell

display(HTML("<style>.container { width:98% !important; }</style>"))
%load_ext autoreload  
%autoreload 2
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import os
import gc
import sys 
import time
import tomllib
import logging
from datetime import datetime
from pathlib import Path
from collections import Counter

import wandb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.autonotebook import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import lightning as L

from torchvision import transforms
from torchvision import datasets, transforms
import torchvision.models as models
import torchvision.transforms.functional as TVF
# from torchvision import transforms
# from torchinfo import summary

from sklearn.model_selection import train_test_split
from abc import ABC, abstractmethod
import torchmetrics
import torchvision.transforms.functional as TVF
from torchmetrics.classification import (MulticlassConfusionMatrix, 
                                         MulticlassCalibrationError,
                                         MulticlassAccuracy, 
                                         MulticlassAUROC, MulticlassF1Score)

/tmp/ipykernel_4519/1220432866.py:17: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
pd.set_option('display.width',180)
torch.set_printoptions(linewidth=180)
np.set_printoptions(linewidth=180)

In [4]:
os.environ['WANDB_NOTEBOOK_NAME']  = f"{(os.path.basename(os.environ.get('JPY_SESSION_NAME')))}"
SESSION_NAME  = f"{(os.path.basename(os.environ.get('JPY_SESSION_NAME')))}"

# <font style="color:green">Data Loader [10 Points]</font>

In this section, you have to write a class or methods, which will be used to get training and validation data loader.

You need to write a custom dataset class to load data.

**Note; There is   no separate validation data. , You will thus have to create your own validation set, by dividing the train data into train and validation data. Usually, we do 80:20 ratio for train and validation, respectively.**


For example:

```python
class KenyanFood13Dataset(Dataset):
    """
    
    """
    
    def __init__(self, *args):
    ....
    ...
    
    def __getitem__(self, idx):
    ...
    ...
    

```


```python
def get_data(args1, *args):
    ....
    ....
    return train_loader, test_loader
```

In [5]:
class KenyanFoodDataset(Dataset):
    """
    This custom dataset class takes root directory and train flag, 
    and returns dataset training dataset if train flag is true 
    else it returns validation dataset.
    """
    
    def __init__(self, data_dict, training = True, transform=None):
        
        """
        init method of the class.
        
         Parameters:
         
         data_dict (pandas Dataframe): path of root directory.  
         training (tuple): Data split between training and validation data.
         image_shape (int or tuple or list): [optional] int or tuple or list. Default is None. 
                                             If it is not None image will resize to the given shape.       
         transform (method): method that will take PIL image and transform it.
         
        """
        # set transform attribute
        self.transform = transform
        self.data_dict = data_dict
        self.training = training
        print(f" Dataset initialized with {len(self.data_dict)} samples")
        print(f" Sample data: \n {self.data_dict.head()}")
    def __len__(self):
        """
        return length of the dataset
        """
        return len(self.data_dict)
    
    
    def __getitem__(self, idx):
        """
        For given index, return images with resize and preprocessing.
        """
        image = Image.open(self.data_dict['image_path'][idx]).convert("RGB")
        
        if self.transform is not None:
            image = self.transform(image)
        if self.training:
            return image, self.data_dict['label'][idx] 
        else:
            # print(str(self.data_dict['id'][idx]))
            return image, str(self.data_dict['id'][idx])


In [6]:
class KenyanFoodDataModule(L.LightningDataModule):
    
    def __init__(self, 
                 data_root   : str, 
                 batch_size  : int = 12, 
                 num_workers : int = 4, 
                 image_shape : int = 512,
                 shuffle     : bool = False,
                 pin_memory  : bool = False,
                 augmentation: str = None,
                 class_boost : np.array = None, 
                 seed = 42):
        super().__init__()
        print(" DataModule Initialization Started")
        self.data_root = Path(data_root)
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.shuffle = shuffle
        self.augmentation = augmentation.lower() if augmentation is not None else None
        self.pin_memory = pin_memory
        self.seed = seed
        
        # set image_resize attribute
        if image_shape is not None:
            if isinstance(image_shape, int):
                self.image_shape = (image_shape, image_shape)
            elif isinstance(image_shape, tuple) or isinstance(image_shape, list):
                assert len(image_shape) == 1 or len(image_shape) == 2, 'Invalid image_shape tuple size'
                if len(image_shape) == 1:
                    self.image_shape = (image_shape[0], image_shape[0])
                else:
                    self.image_shape = image_shape
            else:
                raise NotImplementedError 
        else:
            self.image_shape = image_shape

        self.training_data = pd.read_csv( os.path.join(self.data_root, 'train.csv'), engine='python')
        self.label_names = list(self.training_data['class'].unique())
        self.label_names.sort()
        self.name_to_label = {x:i for i,x in enumerate(self.label_names)}
        self.label_to_name = {i:x for x,i in self.name_to_label.items()}
        if class_boost is not None:
            assert len(class_boost) == len(self.label_names), "class_boost length should be same as number of classes"
            self.class_boost = torch.Tensor(class_boost)
        else:
            self.class_boost = torch.ones(len(self.label_names))

        self._set_augmentation_strategy()

        # print(f" class names  : {self.label_names}")
        print(f" name_to_label: {self.name_to_label}")
        print(f" label to name: {self.label_to_name}")
        print(f" image shape  : {self.image_shape}")
        print(f" batch_size   : {self.batch_size}")
        print(f" class boost  : {self.class_boost}")
        print(f" num_workers  : {self.num_workers}")
        print(f" seed         : {self.seed}")
        print(f" augmentation : {self.augmentation}")
        print(f" training augmentations: \n {self.aug_transforms}")
        print(" DataModule Initialization Completed")
    
    def prepare_data(self):
        pass
        
    def setup(self, stage=None):
        """Setup datasets for each stage (fit, validate, test, predict)"""
        
        print(f" DataModule Setup started - stage {stage}")
        #-----------------------------------------------------------------
        # We use ImageNet normalization mean/std when fine-tuning/transfer learning
        #-----------------------------------------------------------------

        self.test_data = pd.read_csv( os.path.join(self.data_root, 'test.csv'), engine='python')
        self.test_data['image_path'] = self.test_data['id'].map(lambda x: f"{os.path.join(self.data_root,str(x))}.jpg")        
        
        self.training_data['label'] = self.training_data['class'].map(lambda x: self.name_to_label[x])
        self.training_data['image_path'] = self.training_data['id'].map(lambda x: f"{os.path.join(self.data_root,str(x))}.jpg")        
        
        self.train_split, self.val_split, y_train, y_val = train_test_split(
                 self.training_data, self.training_data['label'], test_size=0.15, 
                 stratify=self.training_data['label'], random_state=self.seed)

        self.train_split.reset_index(inplace=True)
        self.val_split.reset_index(inplace=True)
        
        ## Create weighted sampler
        
        _class_counts = Counter(self.train_split['label'])
        self.class_counts = { x: _class_counts[x] for x in sorted(_class_counts)}
        self.class_weights = torch.Tensor([1.0/self.class_counts[x] for x in sorted(self.class_counts)])
        print(f"    class counts: {self.class_counts}")
        print(f"   class weights: {self.class_weights}")
        if self.class_boost is not None:
            self.class_weights = self.class_weights * self.class_boost
            print(f" boosted weights: {self.class_weights}")
        
        sample_weights =[self.class_weights[x] for x in self.train_split.label]
        self.weighted_sampler = WeightedRandomSampler(
            weights= sample_weights,
            num_samples=len(self.train_split),
            replacement=True,
            generator = torch.Generator().manual_seed(self.seed)
        )
                
        # train_data_path = os.path.join(self.data_root, "training")
        # val_data_path = os.path.join(self.data_root, "validation")

        self.train_dataset = KenyanFoodDataset(data_dict=self.train_split,
                                                 training=True,
                                                 transform=self.aug_transforms)

        self.val_dataset = KenyanFoodDataset(data_dict=self.val_split,
                                                 training=True,
                                                 transform=self.common_transforms)

        self.test_dataset = KenyanFoodDataset(data_dict=self.test_data,
                                                 training=False,
                                                 transform=self.common_transforms)
        
        # Apply different transforms to each split
        # if stage == "fit" or stage is None:
        #     self.train_dataset = self._apply_transform(train_dataset, train_transforms)
        #     self.val_dataset = self._apply_transform(val_dataset, eval_transforms)
        
        # if stage == "test" or stage is None:
        #     self.test_dataset = self._apply_transform(test_dataset, eval_transforms)
        print(" DataModule Setup completed")

    def _set_augmentation_strategy(self):
        mean = [0.485, 0.456, 0.406]
        std = [0.229, 0.224, 0.225]
        
        self.preprocess_transforms = transforms.Compose(
            [transforms.Resize([x+32 for x in self.image_shape]), 
             transforms.CenterCrop(self.image_shape), 
             transforms.ToTensor(),
            ]
        )

        self.common_transforms = transforms.Compose([
             transforms.Resize([x+32 for x in self.image_shape]), 
             transforms.CenterCrop(self.image_shape), 
             transforms.ToTensor(),
             transforms.Normalize(mean, std)
        ])
        
        match self.augmentation:
            case None:
                self.aug_transforms = self.common_transforms
            case 'v1':
                self.aug_transforms = transforms.Compose([
                        transforms.RandomResizedCrop(self.image_shape, scale=(0.8, 1.0)),
                        transforms.RandomHorizontalFlip(p=0.5),
                        transforms.RandomRotation(degrees=5),
                        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                        transforms.RandomAffine(degrees = 0, translate=(0.1,0.1)),
                        transforms.ToTensor(),
                        transforms.Normalize(mean, std)
                    ])
            case 'v5':   ## adds RandomErasing to v1
                self.aug_transforms = transforms.Compose([    
                        transforms.RandomResizedCrop(self.image_shape, scale=(0.8, 1.0)),
                        transforms.RandomHorizontalFlip(),
                        transforms.RandomRotation(15),
                        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                        transforms.RandomAffine(degrees = 0, translate=(0.1,0.1)),               
                        transforms.ToTensor(),
                        transforms.Normalize(mean, std),     
                        transforms.RandomErasing(p=0.5, scale=(0.02, 0.5), ratio=(0.3,1.2),value=0),
                        # transforms.RandomErasing(p=0.5, scale=(0.02, 0.25), ratio=(0.3,1.2),value=0),
                    ])    

            case 'v6':   ## adds AugMix to v1
                self.aug_transforms = transforms.Compose([   
                        transforms.RandomResizedCrop(self.image_shape, scale=(0.8, 1.0)),
                        transforms.RandomHorizontalFlip(),
                        transforms.RandomRotation(15),
                        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                        transforms.RandomAffine(degrees = 0, translate=(0.1,0.1)),            
                        transforms.AugMix(severity= 3),
                        transforms.ToTensor(),
                        transforms.Normalize(mean, std),
                        transforms.RandomErasing(p=0.30, scale=(0.02, 0.25), ratio=(0.3,1.2),value=0),
                    ])                            
            case 'v2':  ## Randomly applies one of the augmentations in the list to each image
                augmentation_options=[
                        # transforms.ElasticTransform(alpha=100.0, sigma = 5.0),
                        transforms.RandomResizedCrop(self.image_shape, scale=(0.8, 1.0)),
                        transforms.RandomHorizontalFlip(p=0.5),
                        transforms.RandomRotation(degrees=15),
                        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                        transforms.RandomAffine(degrees = 0, translate=(0.1,0.1)),            
                        # transforms.AugMix(severity= 3),
                        ]
                self.aug_transforms = transforms.Compose([
                        transforms.RandomResizedCrop(self.image_shape, scale=(0.7, 1.0)),
                        transforms.RandomChoice(transforms=augmentation_options), 
                        transforms.ToTensor(),
                        transforms.Normalize(mean, std),
                        transforms.RandomErasing(p=0.25, scale=(0.02, 0.5), ratio=(0.3,1.2),value=0),
                    ])

            case _:
                raise RuntimeError(' Wrong augmentation version')
        
        return 
                        
    def train_dataloader(self):
        # train loader
        train_loader = DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            sampler=self.weighted_sampler,
            num_workers=self.num_workers,
            pin_memory = self.pin_memory,
            persistent_workers = True,
            drop_last = False
            # shuffle=self.shuffle,
            # in_order = False
        )
        return train_loader

    def val_dataloader(self):
        # validation loader
        val_loader = DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=2,
            pin_memory = self.pin_memory,
            persistent_workers = True,
            drop_last = False
        )
        return val_loader

    def test_dataloader(self):
        # validation loader
        test_loader = DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory = self.pin_memory
        )
        return test_loader      
        

# <font style="color:green">Configuration [5 Points]</font>

**Define your configuration here.**

For example:


```python
@dataclass
class TrainingConfiguration:
    '''
    Describes configuration of the training process
    '''
    batch_size: int = 10 
    epochs_count: int = 50  
    init_learning_rate: float = 0.1  # initial learning rate for lr scheduler
    log_interval: int = 5  
    test_interval: int = 1  
    data_root: str = "/kaggle/input/opencv-pytorch-project-2-classification-round-3" 
    num_workers: int = 2  
    device: str = 'cuda'  
    
```

In [7]:
from dataclasses import dataclass

In [8]:
@dataclass
class SystemConfiguration:
    '''
    Describes the common system setting needed for reproducible training
    '''
    seed: int = 21  # seed number to set the state of all random number generators
    cudnn_benchmark_enabled: bool = True  # enable CuDNN benchmark for the sake of performance
    cudnn_deterministic: bool = True  # make cudnn deterministic (reproducible training)


In [9]:

@dataclass
class TrainingConfiguration:
    '''
    Describes configuration of the training process
    '''
    augmentation: str = ""
    batch_size: int = 4      # 10 
    class_boost: np.array = None
    data_root: str = "./images" 
    device: str = 'cuda'  
    dropout_rate:float = 0.0
    early_stopping: bool = False
    epochs_count: int = 20    # 50  
    image_shape: int = 224,
    initialization: str = None
    init_learning_rate: float = 0.001  ## 0.1  # initial learning rate for lr scheduler
    last_epoch : int = -1
    log_interval: int = 5  
    logs_root: str = "./logs" 
    lr_factor: float = 1.0   ## LR reduce factor for ReduceLR on Plateau
    model_name:str = 'unknown'
    notebook_name: str = ""
    num_workers: int = 2  
    optimizer: str = ""
    project_name: str =""
    run_name: str =""
    session_name: str = ""
    scheduler: str = ""
    subset_size: float = 1.00
    test_interval: int = 1  
    tuning_layers : str = ""
    use_tensorboard: bool = False
    warmup_iters: int = 0
    weight_decay: float = 1.0e-04

In [10]:
@dataclass
class InferenceConfiguration:
    '''
    Describes configuration of the training process
    '''
    batch_size: int = 18      # 10 
    data_root: str = "./images" 
    device: str = 'cuda'  
    epochs_count: int = 20    # 50  
    image_shape: int = 224,
    initialization: str = None
    logs_root: str = "./logs" 
    model_name:str = 'unknown'
    notebook_name: str = ""
    num_workers: int = 1
    project_name: str =""
    run_name: str =""
    session_name: str = ""
    test_interval: int = 1  

## <font style="color:green">3. Evaluation Metric [10 Points]</font>

**Define methods or classes that will be used in model evaluation. For example, accuracy, f1-score etc.**

In [11]:
@dataclass
class Metrics():
    '''
    Metrics used in the training process
    '''
    best_loss_ep = -1
    best_loss = torch.tensor(np.inf)
    best_loss_acc = 0.0
    best_acc = 0.0

    # epoch train/test loss
    train_loss = np.array([])
    train_acc  = np.array([])
    # train_acc_2  = np.array([])

    # epch train/test accuracy
    val_loss = np.array([])
    val_acc  = np.array([]) 
    val_acc_macro  = np.array([]) 
    val_cm = np.array([]) 
    val_aucroc = np.array([])
    val_mcce = np.array([])
    
    test_loss = np.array([])
    test_acc  = np.array([]) 

    #f1 metrics
    train_f1 = np.array([])
    val_f1 = np.array([])



# <font style="color:green">Train and Validation [5 Points]</font>


**Write the methods or classes to be used for training and validation.**

In [12]:
def train(train_config: TrainingConfiguration, 
            model: nn.Module, 
            optimizer: torch.optim.Optimizer,
            train_loader: torch.utils.data.DataLoader,
            epoch_idx: int
        ) -> None:
    """
    train the model for one epoch
    
    Args:
        train_config (TrainingConfiguration): training configuration
        model (nn.Module): model to be trained
        optimizer (torch.optim.Optimizer): optimizer to use
        train_loader (torch.utils.data.DataLoader): training data loader
        epoch_idx (int): current epoch index
    
    Returns:
        epoch_loss (float): average loss for the epoch
        epoch_acc (float): average accuracy for the epoch
        trn_acc (float): torchmetrics accuracy for the epoch    
    """
    # change model in training mood
    model.train()

    # to get batch loss
    batch_loss = np.array([])

    batch_acc = torchmetrics.classification.Accuracy(task="multiclass",
                                                     average = 'micro',
                                                     num_classes=13).to(train_config.device)
    progress_bar = tqdm(train_loader, desc='Training', leave=False)
    
    for batch_idx, (data, target) in enumerate(train_loader):

        # clone target
        # indx_target = target.clone()
        # send data to device (its is medatory if GPU has to be used)
        data = data.to(train_config.device)
        # send target to device
        target = target.to(train_config.device)

        # reset parameters gradient to zero
        optimizer.zero_grad()

        # forward pass to the model
        output = model(data)

        # cross entropy loss
        loss = F.cross_entropy(output, target)

        # find gradients w.r.t training parameters
        loss.backward()
        # Update parameters using gardients
        optimizer.step()

        batch_loss = np.append(batch_loss, [loss.item()])
        avg_loss = batch_loss.mean()
        batch_acc.update(output, target)
        
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'avg_loss': f'{avg_loss:.4f}'
        })        
        progress_bar.update(1)
    
    progress_bar.close()
    epoch_loss = batch_loss.mean()
    epoch_acc = batch_acc.compute().cpu().item()

    return epoch_loss, epoch_acc

In [13]:

def validate(train_config: TrainingConfiguration,
            model: nn.Module,
            val_loader: torch.utils.data.DataLoader,
            metrics: Metrics
            ) -> float:
    """
    validate the model on validation dataset
    Args:
        train_config (TrainingConfiguration): training configuration
        model (nn.Module): model to be validated
        val_loader (torch.utils.data.DataLoader): validation data loader    
    Returns:
        val_loss (float): average validation loss
        accuracy (float): average validation accuracy
        val_acc_2 (float): torchmetrics accuracy for the validation dataset
    """
    model.eval()
    
    _val_loss = 0.0
    # count_corect_predictions = 0
    _val_cm = MulticlassConfusionMatrix(num_classes=13,
                                        normalize=None).to(train_config.device)
    # MulticlassAccuracy:
    #
    # average: micro: Sum statistics over all labels
    #          macro: Calculate statistics for each label and average them
    #       weighted: calculates statistics for each label and computes weighted 
    #                 average using their support    
    _val_acc = MulticlassAccuracy(num_classes=13, 
                                  average="micro").to(train_config.device)
    _val_acc_macro = MulticlassAccuracy(num_classes=13,
                                 average='macro').to(train_config.device)    
    _val_aucroc = MulticlassAUROC(num_classes=13, 
                                  average="weighted", 
                                  thresholds=10).to(train_config.device)
    _val_f1 = MulticlassF1Score(num_classes=13, 
                                average="weighted").to(train_config.device)
    _val_mcce = MulticlassCalibrationError(num_classes = 13).to(train_config.device)
    
    for data, target in val_loader:
        
        data = data.to(train_config.device)
        target = target.to(train_config.device)

        with torch.no_grad():
            preds = model(data)
        # print(f" shapes: {data.shape}   target: {target.shape}  preds: {preds.shape}")
        
        # add loss for each mini batch
        _val_loss += F.cross_entropy(preds, target).item()
        _val_acc.update(preds,target)
        _val_acc_macro.update(preds,target)
        _val_cm.update(preds, target)
        _val_aucroc.update(preds, target)
        _val_f1.update(preds, target)
        _val_mcce.update(preds, target)
        
        # Score to probability using softmax
        # probs = F.softmax(preds, dim=1)
        # # get the index of the max probability
        # pred_labels = probs.data.max(dim=1)[1] 
        # count_corect_predictions += pred_labels.cpu().eq(indx_target).sum()

        
    # average over number of mini-batches
    # accuracy = ( count_corect_predictions / len(val_loader.dataset)).item()
    metrics.val_loss    = np.append(metrics.val_loss  , _val_loss / len(val_loader))
    metrics.val_acc     = np.append(metrics.val_acc   , _val_acc.compute().cpu().numpy())
    metrics.val_acc_macro= np.append(metrics.val_acc_macro   , _val_acc_macro.compute().cpu().numpy())
    metrics.val_cm      = np.append(metrics.val_cm    , _val_cm.compute().cpu().numpy())
    metrics.val_aucroc  = np.append(metrics.val_aucroc, _val_aucroc.compute().cpu().numpy())
    metrics.val_f1      = np.append(metrics.val_f1    , _val_f1.compute().cpu().numpy())
    metrics.val_mcce    = np.append(metrics.val_mcce  , _val_mcce.compute().cpu().numpy())
    val_loss   = _val_loss / len(val_loader)
    val_acc    = _val_acc.compute().cpu().numpy()
    return val_loss, val_acc


In [14]:

def main(model, 
         optimizer, 
            scheduler=None, 
            system_configuration=None, 
            train_config=None, 
            metrics = None,
            train_loader = None, 
            val_loader = None,
            wandb_session = None,
            tb_writer = None
        ):
    
    # system configuration
    setup_system(system_configuration)

    # if GPU is available use training config, 
    train_config.device = "cuda" if torch.cuda.is_available() else "cpu"
 
    print(f" moving model to {train_config.device}")
    model.to(train_config.device)

    # Calculate Initial Test Loss
    init_val_loss, init_val_acc = validate(train_config, model, val_loader, metrics)
    print(f" Model run: {train_config.run_name}")
    print(f" Model: {train_config.model_name}   Batch Size: {train_config.batch_size}   "
          f"Init LR: { train_config.init_learning_rate}  Epochs in this run: {train_config.epochs_count}")
    print(" Initial Test Loss : {:.6f} \t Initial Test Accuracy : {:.3f} %\n".format(
            init_val_loss, init_val_acc*100))

    # trainig time measurement
    print(" Epoch     Trn Loss    Trn Acc      Val Loss    Val Acc      macro       Elpsd(s)     (s/ep)    (s/bat)    ETA (s)      LR        F1    Calib")
            # 6       0.269182     0.9134      0.802107     0.7533     0.7263         364.22      52.03       0.90     3538.2    9.8e-03   0.7528  0.1000   Acc Imp: 75.33% .   Loss Imp: 0.802107 .    
    t_begin = time.time()
    start_epoch = train_config.last_epoch + 1
    end_epoch = train_config.last_epoch + 1 + train_config.epochs_count

    patience_counter = 0
    
    for epoch in range(start_epoch, end_epoch):
        train_config.last_epoch = epoch
        
        # Train
        train_loss, train_acc = train(train_config, model, optimizer, train_loader, epoch)
        metrics.train_loss = np.append(metrics.train_loss, [train_loss])
        metrics.train_acc = np.append(metrics.train_acc, [train_acc])

        elapsed_time = time.time() - t_begin
        speed_epoch = elapsed_time / (epoch + 1)
        speed_batch = speed_epoch / len(train_loader)
        eta = speed_epoch * train_config.epochs_count - elapsed_time

        # Validate
        if epoch % train_config.test_interval == 0:
            
            val_loss, val_acc = validate(train_config, model, val_loader, metrics)

            print(" {:3d}   {:12.6f}  {:9.4f}  {:12.6f}  {:9.4f}  {:9.4f}     {:10.2f}    {:7.2f}    {:7.2f}    {:7.1f}    {:.1e}   {:.4f}  {:.4f}  ".format(
                    epoch, train_loss, train_acc, val_loss, val_acc, metrics.val_acc_macro[-1], elapsed_time, speed_epoch, speed_batch, 
                    eta, scheduler.get_last_lr()[0], metrics.val_f1[-1], metrics.val_mcce[-1]), end='')
 
            if val_acc > metrics.best_acc:
                metrics.best_acc = val_acc
                metrics.best_acc_loss = val_loss
                metrics.best_acc_ep = epoch
                print(f"Acc: {metrics.best_acc*100:.2f}% ", end="")
                save_model(model, device=train_config.device, 
                           model_file_name=train_config.checkpoint_name+"_bestacc.pt")
                
            if val_loss < metrics.best_loss:
                metrics.best_loss = val_loss
                metrics.best_loss_acc = val_acc
                metrics.best_loss_ep = epoch
                print(f"Loss: {metrics.best_loss:.6f} ", end="")
                save_model(model, device=train_config.device, 
                           model_file_name=train_config.checkpoint_name+"_bestloss.pt")
                patience_counter = 0 
            else:
                # patience_counter +=1
                pass
            
        # Scheduler step
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
                print(f" {scheduler.cooldown_counter:2d}-{scheduler.num_bad_epochs}", end ='')
            else:
                scheduler.step()
        print()
                
        if wandb_session is not None:
            wandb_session.log({'trn_loss': train_loss, 'trn_acc': train_acc, 
                               'val_loss':val_loss, 'val_acc' : val_acc, 
                               'best_loss':metrics.best_loss, 'best_acc':metrics.best_acc,
                               'best_loss_ep':metrics.best_loss_ep, 'best_acc_ep': metrics.best_acc_ep,
                               'lr[0]':scheduler.get_last_lr()[0]})
            
        if tb_writer is not None:
            tb_write_metrics(tb_writer, metrics, epoch)
            
        # Early stop based on loss
        if train_config.early_stopping and (patience_counter >= train_config.early_stopping):
            print(f" ***** Early stopping threshold met {train_config.early_stopping} *****")
            print(f" ***** Early stopping at epoch {epoch} *****")
            break
    # end of epoch -------------------
        
    print()
    print("-"*100)    
    print("  Total time: {:.2f}  \n" \
          "  Best Loss Epoch: {:4d}     Best Loss:   {:.6f}      Best Loss Accuracy: {:.3f} % \n" \
          "  Best Acc  Epoch: {:4d}     Best Accuracy: {:.3f} %    Best Accuracy Loss: {:.6f}".format(
             time.time() - t_begin, 
             metrics.best_loss_ep, metrics.best_loss, metrics.best_loss_acc*100,
             metrics.best_acc_ep , metrics.best_acc * 100, metrics.best_acc_loss))
    print("-"*100)    

    return model, metrics 

# <font style="color:green">Model [5 Points]</font>

**Define your model in this section.**

**You are allowed to use any pre-trained model.**

In [15]:
class PretrainedResNet(nn.Module,ABC):
    """ 
    Abstract base class for pretrained ResNet models with selective fine-tuning.
    """
    def __init__(self, num_classes=13, finetune=None, name = None, weights=None, verbose = False):
        super().__init__()
        # Subclasses must provide these
        model_fn, model_name, weights = self._get_model_config()
        # assert "fc" in finetune, "'fc' must be in fine tuning"
        
        self.name = model_name if name is None else name
        if finetune is not None:
            self.finetune = finetune
            self.tuning_layers = ''.join(
                ['fc'] + sorted([x[-1] for x in self.finetune 
                           if (x[:-1] == 'layer' and x != 'fc')], 
                          reverse=True)
                )
        else:
            self.finetune = []
            self.tuning_layers = ''
        
        # Load pretrained model
        self.resnet = model_fn(weights=weights)
        
        # Replace final layer
        last_layer_in = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(last_layer_in, num_classes)
        
        # Freeze all parameters
        for param in self.resnet.parameters():
            param.requires_grad = False
        
        # Unfreeze specified layers
        for ft_layer in self.finetune:
            if verbose: 
                print(f" turn on gradients for layer {ft_layer}")
            layer = getattr(self.resnet, ft_layer)
            
            for param in layer.named_parameters():
                param[1].requires_grad = True
                if verbose:
                    print(f" param : {param[0]:35s}  Shape: {param[1].shape}  "
                          f"Requires Grad Set to True", end='')
                
    @abstractmethod
    def _get_model_config(self):
        """Return (model_fn, model_name, default_weights) tuple."""
        pass
        
    def forward(self, x):
        return self.resnet(x)    

In [16]:
class PretrainedResNet18(PretrainedResNet):
    def _get_model_config(self):
        return (
            models.resnet18,
            "resnet18",
            models.ResNet18_Weights.IMAGENET1K_V1
        )

class PretrainedResNet34(PretrainedResNet):
    def _get_model_config(self):
        return (
            models.resnet34,
            "resnet34",
            models.ResNet34_Weights.IMAGENET1K_V1
        )

class PretrainedResNet50(PretrainedResNet):
    def _get_model_config(self):
        return (
            models.resnet50,
            "resnet50",
            models.ResNet50_Weights.IMAGENET1K_V2
        )
        
class PretrainedResNet101(PretrainedResNet):
    def _get_model_config(self):
        return (
            models.resnet101,
            "resnet101",
            models.ResNet101_Weights.IMAGENET1K_V2
        )
        
        

# <font style="color:green">Utils [5 Points]</font>

**Define those methods or classes, which have  not been covered in the above sections.**

In [17]:
def delete_vars(variables = ['model',  'metrics', 'train_config', 'sys_config', 'optimizer', 'scheduler', 'train_loader', 'val_loader', 'data_module']):
    for vr in variables:
        global_vars = globals()
        if vr in global_vars:
            try:
                del globals()[vr]
                print(f" {vr:15s} deleted . . . ")
            except Exception as e:
                print(e)
        else:
            print(f" {vr:15s} NOT defined . . .")
    
    print(f" gc.collect() : {gc.collect()}")
    print(f" Cuda empty cache : {torch.cuda.empty_cache()}")


In [18]:
def setup_system(system_config: SystemConfiguration) -> None:
    torch.manual_seed(system_config.seed)
    if torch.cuda.is_available():
        torch.backends.cudnn_benchmark_enabled = system_config.cudnn_benchmark_enabled
        torch.backends.cudnn.deterministic = system_config.cudnn_deterministic


In [19]:
def save_model(model, device, model_dir='models', model_file_name='cat_dog_panda_classifier.pt'):

    if not os.path.exists(model_dir):
        os.makedirs(model_dir)

    model_path = os.path.join(model_dir, model_file_name)

    # make sure you transfer the model to cpu.
    if device == 'cuda':
        model.to('cpu')

    # save the state_dict
    torch.save(model.state_dict(), model_path)

    if device == 'cuda':
        model.to('cuda')
    return


def load_model(model, model_dir='models', model_file_name='cat_dog_panda_classifier.pt'):
    model_path = os.path.join(model_dir, model_file_name)

    # loading the model and getting model parameters by using load_state_dict
    model.load_state_dict(torch.load(model_path))
    return model    

In [20]:
def save_checkpoint(model, optimizer, scheduler, metrics, train_config, 
                   model_dir='models', model_file_name='cat_dog_panda_classifier.pt'):

    if not os.path.exists(model_dir):
        os.makedirs(model_dir)

    model_path = os.path.join(model_dir, model_file_name)

    # make sure you transfer the model to cpu.
    if train_config.device == 'cuda':
        model.to('cpu')

    # save the state_dict
    torch.save({'epoch':train_config.last_epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'metrics': metrics,
                'training_config':train_config}
                , model_path)
    print(f" model written to {model_path}")
    if train_config.device == 'cuda':
        model.to('cuda')
    return

In [21]:
def load_checkpoint(ckpt_dir='models', ckpt_file_name='cat_dog_panda_classifier.pt'):

    checkpoint_path = os.path.join(ckpt_dir, ckpt_file_name)
    print(checkpoint_path)
    # make sure you transfer the model to cpu.
    print(f" loading checkpoint from {checkpoint_path}")
    return torch.load(checkpoint_path, weights_only=False)


In [22]:
def wandb_init(args, verbose = False):
    """
    args : args to log
        args.exp_id
        args.exp_name
        args.exp_description
        args.project_name
        log_metrics: Metrics to log
    """
    log_metrics = []
    verbose=False
 
    # if wandb.run is not None:
    #     print(f" End in-flight wandb run . . .")
    #     wandb.finish()
    # else:
    #     print(f" Initiate new W&B job run")
    settings = wandb.Settings(
        show_errors=True,  # Show error messages in the W&B App
        silent=False,      # Disable all W&B console output
        show_warnings=True, # Show warning messages in the W&B App
        show_info=True, 
        console = "redirect",
        # console_multipart = True,
        # console_chunk_max_seconds = 600,
        notebook_name = args.session_name
    )    
    if not hasattr(args,"wandb_id"):
        # opt['exp_id'] = wandb.util.generate_id()
        # print_dbg(f"{opt['exp_id']}, {opt['exp_name']}, {opt['project_name']}", verbose) 
        wandb_run = wandb.init(project=args.project_name, 
                            entity="kbardool", 
                            name = args.run_name,
                            resume="never" ,
                            settings = settings)
        args.wandb_id = wandb_run.id
    else:
        wandb_run = wandb.init(project=args.project_name, 
                            entity="kbardool", 
                            id = args.wandb_id, 
                            resume="must", 
                            settings = settings)
        args.wandb_id = wandb_run.id
        if wandb_run.project != '':
            args.project_name = wandb_run.project
        else:
            wandb_run.project = args.project_name 
            
        if wandb_run.name != '':
            args.exp_name = wandb_run.name
        else:
            wandb_run.name = args.exp_name 
            
        if wandb_run.notes != '':
            args.exp_description = wandb_run.notes
        else:
            wandb_run.notes = args.exp_description 
        
    wandb.config.update(args,allow_val_change= True)
    
    for metric in log_metrics:
        wandb.define_metric(metric, summary="last")
    # assert wandb.run is None, "Run is still running"
    if verbose:
        print(f" WandB Initialization -----------------------------------------------------------\n"
              f" PROJECT NAME: {wandb_run.project}\n"
              f" RUN ID      : {wandb_run.id} \n"
              f" RUN NAME    : {wandb_run.name}\n"     
              f" RUN NOTES   : {wandb_run.notes}\n"     
              f" --------------------------------------------------------------------------------")
    return wandb_run
 

In [23]:
def tb_write_metrics(tb_writer, mtrcs, epoch):
    # add scalar (loss/accuracy) to tensorboard
    tb_writer.add_scalar('Loss/Train', mtrcs.train_loss[-1], epoch)
    tb_writer.add_scalar('Accuracy/Train', mtrcs.train_acc[-1], epoch)
    
    # add scalar (loss/accuracy) to tensorboard
    tb_writer.add_scalar('Loss/Val', mtrcs.val_loss[-1], epoch)
    tb_writer.add_scalar('Accuracy/Val', mtrcs.val_acc[-1], epoch)
    
    # add scalars (loss/accuracy) to tensorboard
    tb_writer.add_scalars('Loss/train-val', {'train': mtrcs.train_loss[-1], 
                                             'validation': mtrcs.val_loss[-1]}, epoch)
    tb_writer.add_scalars('Accuracy/train-val', {'train': mtrcs.train_acc[-1], 
                                                 'validation': mtrcs.val_acc[-1]}, epoch)

# <font style="color:green">Experiment [5 Points]</font>

**Choose your optimizer and LR-scheduler and use the above methods and classes to train your model.**

In [24]:
timestamp = datetime.now().strftime('%Y_%m_%d_%H:%M:%S')
logger = logging.getLogger(__name__) 
logLevel = os.environ.get('LOG_LEVEL', 'INFO').upper()
FORMAT = '%(asctime)s - %(name)s - %(levelname)s: - %(message)s'
logging.basicConfig(level="INFO", format= FORMAT)

logger.info(f" Excution started : {timestamp} ")
logger.info(f" Pytorch version  : {torch.__version__}  \t\t Number of threads: {torch.get_num_threads()}")
logger.info(f" WandB version    : {wandb.__version__}  \t\t Pandas version: {pd.__version__}  ")

2026-03-02 19:57:47,689 - __main__ - INFO: -  Excution started : 2026_03_02_19:57:47 
2026-03-02 19:57:47,691 - __main__ - INFO: -  Pytorch version  : 2.9.1  		 Number of threads: 6
2026-03-02 19:57:47,692 - __main__ - INFO: -  WandB version    : 0.23.1  		 Pandas version: 2.3.3  


#### delete any objects residing on Cuda

In [25]:
delete_vars()
gc.collect()

 model           NOT defined . . .
 metrics         NOT defined . . .
 train_config    NOT defined . . .
 sys_config      NOT defined . . .
 optimizer       NOT defined . . .
 scheduler       NOT defined . . .
 train_loader    NOT defined . . .
 val_loader      NOT defined . . .
 data_module     NOT defined . . .
 gc.collect() : 232
 Cuda empty cache : None


0

### Set training parameters 

These will be use to set training_configuration

In [26]:
settings_toml = """
BATCH_SIZE   = 96
EPOCHS_COUNT = 60
WEIGHT_DECAY = 7.5e-03
LR_FACTOR    = 0.85
LR_PATIENCE  = 3
LR_COOLDOWN  = 1
SCHEDULER    = 'ReduceLROnPlateau'
INIT_LEARNING_RATE = 1.0e-02
NUM_WORKERS   = 6
TUNING_LAYERS = ['fc', 'layer4', 'layer3']
PROJECT_NAME  = 'OpenCV_Pytorch_Week7'
DROPOUT_RATE  = 0.0
LAST_EPOCH    = -1
AUGMENTATION  = 'V5'
WARMUP_ITERATIONS  = 0
OPTIMIZER     = 'SGD'
CLASS_BOOST   = [1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1]
USE_TENSORBOARD = true
"""
# settings_toml

In [27]:
metrics = Metrics()
sys_config = SystemConfiguration()
train_config = TrainingConfiguration()

In [28]:
# print(" Training Configuration : ")
# for k,v in train_config.__dict__.items():
#     print(f"  {k:25s}: {v}")

In [29]:
import tomllib
input_params = tomllib.loads(settings_toml)
# with open('settings_1.toml', "rb") as f:
#     input_params = tomllib.load(f)

for k,v in input_params.items():
    k = k.lower()
    # print(f" {k:25s}  {type(v)}  {v:}    {data.get(k, v)}")
    if k not in train_config.__dict__.keys():
        print(f" '{k}'           not found in training configuration - added . . . ")
    setattr(train_config, k, v)
    print(f"  {k:25s}: {v}")

  batch_size               : 96
  epochs_count             : 60
  weight_decay             : 0.0075
  lr_factor                : 0.85
 'lr_patience'           not found in training configuration - added . . . 
  lr_patience              : 3
 'lr_cooldown'           not found in training configuration - added . . . 
  lr_cooldown              : 1
  scheduler                : ReduceLROnPlateau
  init_learning_rate       : 0.01
  num_workers              : 6
  tuning_layers            : ['fc', 'layer4', 'layer3']
  project_name             : OpenCV_Pytorch_Week7
  dropout_rate             : 0.0
  last_epoch               : -1
  augmentation             : V5
 'warmup_iterations'           not found in training configuration - added . . . 
  warmup_iterations        : 0
  optimizer                : SGD
  class_boost              : [1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1]
  use_tensorboard          : True


### Data Loader

In [30]:
data_module = KenyanFoodDataModule(
    data_root = train_config.data_root, 
    batch_size = train_config.batch_size, 
    num_workers = train_config.num_workers, 
    image_shape = train_config.image_shape,
    class_boost = train_config.class_boost,
    augmentation = train_config.augmentation,
    pin_memory=True,seed=84
)
data_module.prepare_data()
data_module.setup()
train_loader = data_module.train_dataloader()
val_loader = data_module.val_dataloader()

 DataModule Initialization Started
 name_to_label: {'bhaji': 0, 'chapati': 1, 'githeri': 2, 'kachumbari': 3, 'kukuchoma': 4, 'mandazi': 5, 'masalachips': 6, 'matoke': 7, 'mukimo': 8, 'nyamachoma': 9, 'pilau': 10, 'sukumawiki': 11, 'ugali': 12}
 label to name: {0: 'bhaji', 1: 'chapati', 2: 'githeri', 3: 'kachumbari', 4: 'kukuchoma', 5: 'mandazi', 6: 'masalachips', 7: 'matoke', 8: 'mukimo', 9: 'nyamachoma', 10: 'pilau', 11: 'sukumawiki', 12: 'ugali'}
 image shape  : (224, 224)
 batch_size   : 96
 class boost  : tensor([1., 1., 1., 1., 3., 1., 1., 1., 1., 1., 1., 1., 1.])
 num_workers  : 6
 seed         : 84
 augmentation : v5
 training augmentations: 
 Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2), hue=(-0.1, 

### Define Model

In [31]:
model = PretrainedResNet50(finetune=train_config.tuning_layers)

### Create run_name for wandb 

In [32]:
do = f"_p{train_config.dropout_rate:3.1f}" if train_config.dropout_rate != 0.0 else ""
bs = f"BS_{train_config.batch_size:03d}"
lr = f"LR_{ train_config.init_learning_rate:.1e}"
wd = f"wd{train_config.weight_decay:.1e}"
wp = f"wp_{train_config.warmup_iters:02d}"
print(model.name, model.tuning_layers, model.finetune)
print(SESSION_NAME)

train_config.notebook_name = SESSION_NAME
train_config.session_name = SESSION_NAME
train_config.model_name = model.name


train_config.run_name = f"{train_config.model_name}_{bs}_{lr}_{model.tuning_layers}_{wd}_{wp}"
print(f" run name: {train_config.run_name}")

resnet50 fc43 ['fc', 'layer4', 'layer3']
Project2_Kaggle_Competition_Classification-Round3.ipynb
 run name: resnet50_BS_096_LR_1.0e-02_fc43_wd7.5e-03_wp_00


### Tensorboard Summary Writer

In [33]:
if train_config.use_tensorboard:
    tb_writer = SummaryWriter(
        log_dir = os.path.join(train_config.logs_root, train_config.run_name),
        comment = "week 7 Competition"
    )


### Optimizer & Scheduler 

In [34]:
match train_config.optimizer:
    case "SGD":
        optimizer = optim.SGD(model.parameters(),
                              lr= train_config.init_learning_rate,
                              momentum=0.9,
                              dampening=0,
                              weight_decay=train_config.weight_decay,
                              nesterov=True)
    case 'Adam':
        optimizer = optim.Adam(model.parameters(),
                               lr= train_config.init_learning_rate,
                               weight_decay=train_config.weight_decay)
    case _:
        print(f"Unknown optimizer type: {train_config.optimizer}")



If warmup_iters == 0 , no warm-up iterations will occur 

In [35]:
match train_config.scheduler:
    case "ReduceLROnPlateau":
        scheduler  = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', 
                                                  factor=train_config.lr_factor,
                                                  patience=train_config.lr_patience, 
                                                  cooldown=train_config.lr_cooldown, 
                                                  threshold=0.00001, 
                                                  threshold_mode='rel', 
                                                  min_lr=0, eps=1e-08)
    case "Cosine":
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_config.epochs_count - train_config.warmup_iters)
    case _:
        print(f"Unknown scheduler type: {train_config.scheduler}")

if train_config.scheduler == "Cosine":    
    start_factor = 1.0 if train_config.warmup_iters == 0 else 0.1
    warmup_scheduler = lr_scheduler.LinearLR(optimizer, 
                                             start_factor=start_factor,   # Start at 10% of base LR 
                                             total_iters=train_config.warmup_iters    # Warmup for 5 epochs
                                             # train_config.init_learning_rate/train_config.warmup_iters,  
                                            ) 

    scheduler = optim.lr_scheduler.SequentialLR(optimizer,
        schedulers=[warmup_scheduler, scheduler],
        milestones=[train_config.warmup_iters]  # Switch after epoch 5
    )


### Weights and Bias init (not active in submitted notebook)

In [36]:
wandb_session = wandb_init(train_config)
wandb_session.watch(model, log_freq=100)

# wandb_session = None
train_config.checkpoint_name = train_config.run_name +f"_{wandb_session.id}" if wandb_session else ''

wandb: Currently logged in as: kbardool to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [37]:
train_config.checkpoint_name = train_config.run_name +f"_{wandb_session.id}" if wandb_session else ''
print(train_config.checkpoint_name)

resnet50_BS_096_LR_1.0e-02_fc43_wd7.5e-03_wp_00_40zqk3yc


### Display Training Config Settings

In [38]:
print(" Training Configuration : ")
for k,v in train_config.__dict__.items():
    print(f"  {k:25s}: {v}")

 Training Configuration : 
  augmentation             : V5
  batch_size               : 96
  class_boost              : [1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1]
  data_root                : ./images
  device                   : cuda
  dropout_rate             : 0.0
  early_stopping           : False
  epochs_count             : 60
  image_shape              : (224,)
  initialization           : None
  init_learning_rate       : 0.01
  last_epoch               : -1
  log_interval             : 5
  logs_root                : ./logs
  lr_factor                : 0.85
  model_name               : resnet50
  notebook_name            : Project2_Kaggle_Competition_Classification-Round3.ipynb
  num_workers              : 6
  optimizer                : SGD
  project_name             : OpenCV_Pytorch_Week7
  run_name                 : resnet50_BS_096_LR_1.0e-02_fc43_wd7.5e-03_wp_00
  session_name             : Project2_Kaggle_Competition_Classification-Round3.ipynb
  scheduler                : Red

### Call main training module

In [39]:
model, metrics = main(model, 
                      optimizer=optimizer, 
                      scheduler=scheduler, 
                      system_configuration=sys_config,
                      train_config=train_config, 
                      metrics = metrics,
                      train_loader = train_loader,
                      val_loader  = val_loader,
                      wandb_session = wandb_session,
                      tb_writer=tb_writer
                    )


 moving model to cuda
 Model run: resnet50_BS_096_LR_1.0e-02_fc43_wd7.5e-03_wp_00
 Model: resnet50   Batch Size: 96   Init LR: 0.01  Epochs in this run: 60
 Initial Test Loss : 2.586974 	 Initial Test Accuracy : 6.932 %

 Epoch     Trn Loss    Trn Acc      Val Loss    Val Acc      macro       Elpsd(s)     (s/ep)    (s/bat)    ETA (s)      LR        F1    Calib


Training:   0%|          | 0/58 [00:00<?, ?it/s]

   0       1.967954     0.3759      1.376412     0.5209     0.5434          28.24      28.24       0.49     1666.2    1.0e-02   0.5137  0.1045  Acc: 52.09% Loss: 1.376412   0-0


Training:   0%|          | 0/58 [00:00<?, ?it/s]

   1       1.031731     0.6599      0.942497     0.6789     0.6775          76.65      38.32       0.66     2222.8    1.0e-02   0.6765  0.0537  Acc: 67.89% Loss: 0.942497   0-0


...

  57       0.093240     0.9761      0.753842     0.7798     0.7477        2031.28      35.02       0.60       70.0    2.3e-03   0.7797  0.0710    0-3


Training:   0%|          | 0/58 [00:00<?, ?it/s]

  58       0.092698     0.9741      0.726204     0.7788     0.7361        2065.54      35.01       0.60       35.0    2.3e-03   0.7720  0.0863    1-0


Training:   0%|          | 0/58 [00:00<?, ?it/s]

  59       0.086991     0.9779      0.775148     0.7727     0.7353        2100.06      35.00       0.60        0.0    2.0e-03   0.7686  0.0874    0-0

----------------------------------------------------------------------------------------------------
  Total time: 2105.45  
  Best Loss Epoch:    9     Best Loss:   0.706952      Best Loss Accuracy: 77.370 % 
  Best Acc  Epoch:   53     Best Accuracy: 78.797 %    Best Accuracy Loss: 0.803450
----------------------------------------------------------------------------------------------------


### End Weight & Biases Monitoring, issue final checkpoint

In [61]:
wandb_session.finish()

In [147]:
model_file_name =f"{train_config.checkpoint_name}_final_ep{train_config.last_epoch}.pt"
logger.info(f"Save model to {model_file_name}")

# save_checkpoint(model, optimizer, scheduler, metrics, train_config, 
#                model_dir='models', model_file_name=model_file_name)

2026-03-02 19:50:22,732 - __main__ - INFO: - Save model to unknown_BS_096_LR_1.0e-02_fc43_wd7.5e-03_wp_00_46239ge1_final_ep22.pt


In [ ]:
logger.info(f" Execution ended : {datetime.now().strftime('%Y_%m_%d_%H:%M:%S')} ")

# <font style="color:green">TensorBoard Log Link [5 Points]</font>

**Share your TensorBoard scalars logs link here You can also share (not mandatory) your GitHub link, if you have pushed this project in GitHub.**


Note: In light of the recent shutdown of tensorboard.dev, we have updated the submission requirements for your project. Instead of sharing a tensorboard.dev link, you are now required to upload your generated TensorBoard event files directly onto the lab. As an alternative, you may also include a screenshot of your TensorBoard output within your Jupyter notebook. This adjustment ensures that your data visualization and model training efforts are thoroughly documented and accessible for evaluation.

You are also welcome (and encouraged) to utilize alternative logging services like wandB or comet. In such instances, you can easily make your project logs publicly accessible and share the link with others.

<img src="./assets/Tensorboard_1.png" width=1400>

# <font style="color:green">Kaggle Profile Link [50 Points]</font>

**Share your Kaggle profile link  with us here to score , points in  the competition.**

**For full points, you need a minimum accuracy of `75%` on the test data. If accuracy is less than `70%`, you gain  no points for this section.**


**Submit `submission.csv` (prediction for images in `test.csv`), in the `Submit Predictions` tab in Kaggle, to get evaluated for  this section.**

   ### [https://www.kaggle.com/kbardool/competitions](https://www.kaggle.com/kbardool/competitions)

# <font style="color:green">Weights and Biases Link</font>



  ### [https://wandb.ai/kbardool/OpenCV_Pytorch_Week7?nw=nwuserkbardool](https://wandb.ai/kbardool/OpenCV_Pytorch_Week7?nw=nwuserkbardool)